In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/spam_cleaned.csv')
print(df.shape)
df.head()

(5169, 4)


,label,message,label_num,length
0,ham,"Go until jurong point, crazy.. Available only ...",0,111
1,ham,Ok lar... Joking wif u oni...,0,29
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,155
3,ham,U dun say so early hor... U c already then say...,0,49
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,61


In [2]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')   # newer NLTK versions need this too

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gokul\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gokul\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gokul\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

LOWERCASE

In [3]:
text = "Free Entry in 2 a WKLY Comp to WIN FA Cup"
text = text.lower()
print(text)

free entry in 2 a wkly comp to win fa cup


REMOVE SPECIAL CHARACTERS AND PUNCTUATION

In [4]:
import re

text = "Free entry!!! Call NOW @ 09061701461. £1.50"
text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
print(text)

Free entry Call NOW  09061701461 150


TOKENIZE

In [5]:
from nltk.tokenize import word_tokenize

text = "free entry call now"
tokens = word_tokenize(text)
print(tokens)

['free', 'entry', 'call', 'now']


REMOVE STOPWORDS

In [6]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print(list(stop_words)[:20])   # peek at some stopwords

tokens = ['free', 'entry', 'in', 'a', 'wkly', 'comp', 'to', 'win']
filtered = [word for word in tokens if word not in stop_words]
print(filtered)

['above', 'were', 'why', 'are', 'too', 'do', "it's", 'you', 'i', "we'll", 'here', 'which', 'does', 'ain', 'hasn', 'ours', 'won', 'haven', 'once', 'your']
['free', 'entry', 'wkly', 'comp', 'win']


STEMMING

In [7]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

words = ['running', 'runs', 'ran', 'runner', 'easily', 'fairly']
stemmed = [stemmer.stem(w) for w in words]
print(stemmed)

['run', 'run', 'ran', 'runner', 'easili', 'fairli']


COMBINE ALL TECHNIQUES

In [8]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Initialize once outside the function (faster)
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove non-alphanumeric characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # 3. Tokenize
    tokens = word_tokenize(text)
    
    # 4. Remove stopwords AND short tokens (length <= 2)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    # 5. Stem
    tokens = [stemmer.stem(word) for word in tokens]
    
    # 6. Rejoin into a string
    return ' '.join(tokens)

TEST

In [9]:
sample_spam = df[df['label'] == 'spam']['message'].iloc[0]
print("ORIGINAL:")
print(sample_spam)
print("\nPROCESSED:")
print(preprocess_text(sample_spam))

ORIGINAL:
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

PROCESSED:
free entri wkli comp win cup final tkt 21st may 2005 text 87121 receiv entri questionstd txt ratetc appli 08452810075over18


APPLY ON ENTIRE DATASET

In [10]:
# This may take 20-60 seconds depending on your machine
df['cleaned_message'] = df['message'].apply(preprocess_text)

df[['message', 'cleaned_message']].head(10)

,message,cleaned_message
0,"Go until jurong point, crazy.. Available only ...",jurong point crazi avail bugi great world buff...
1,Ok lar... Joking wif u oni...,lar joke wif oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win cup final tkt 21st ma...
3,U dun say so early hor... U c already then say...,dun say earli hor alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf live around though
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey darl week word back like fun still...
6,Even my brother is not like to speak with me. ...,even brother like speak treat like aid patent
7,As per your request 'Melle Melle (Oru Minnamin...,per request mell mell oru minnaminungint nurun...
8,WINNER!! As a valued network customer you have...,winner valu network custom select receivea 900...
9,Had your mobile 11 months or more? U R entitle...,mobil month entitl updat latest colour mobil c...


SANITY CHECK 

In [11]:
# Are there any empty messages after cleaning?
empty_count = (df['cleaned_message'].str.len() == 0).sum()
print(f"Empty messages after cleaning: {empty_count}")

# What's the average length now vs before?
print(f"\nAvg original length: {df['message'].str.len().mean():.1f}")
print(f"Avg cleaned length:  {df['cleaned_message'].str.len().mean():.1f}")

Empty messages after cleaning: 19

Avg original length: 79.0
Avg cleaned length:  46.1


In [12]:
df.to_csv('../data/spam_preprocessed.csv', index=False)
print("Preprocessed data saved!")

Preprocessed data saved!
